# DemandLens — EDA & ML Notebook
**Retail Price Elasticity & Demand Analytics**

Dataset: `data/retail_store_inventory.csv`

This notebook documents the complete end-to-end pipeline used by the DemandLens application:

| Section | Coverage |
|---------|----------|
| 1. Data Loading & Exploration | CO1 — Python / Pandas / NumPy |
| 2. Preprocessing | CO1 — Cleaning, encoding, outlier removal |
| 3. Feature Engineering | CO1 — Derived business features |
| 4. Exploratory Data Analysis | CO1 — Visualisations |
| 5. Price Elasticity Analysis | CO2 — Log-log regression (PED) |
| 6. ML Model Training & Evaluation | CO3 — Linear Regression, Decision Tree, Random Forest |
| 7. Price Optimisation Demo | CO2 — Grid search over price space |

## 0. Imports & Setup

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.insert(0, '..')   # so `utils` package is importable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Project utilities — same modules the Streamlit app uses
from utils.preprocessing       import load_raw_data, preprocess_data, get_feature_matrix
from utils.feature_engineering import add_business_features, compute_price_elasticity, compute_product_elasticity

print('All imports OK')

## 1. Data Loading & Exploration

In [ ]:
raw = load_raw_data()
print('Shape:', raw.shape)
raw.head()

In [ ]:
raw.info()

In [ ]:
raw.describe()

In [ ]:
print('Missing values:')
print(raw.isnull().sum())
print('\nCategories :', raw['Category'].unique())
print('Regions    :', raw['Region'].unique())
print('Seasonality:', raw['Seasonality'].unique())

## 2. Preprocessing

Steps performed inside `utils/preprocessing.py`:
1. Parse dates → extract Year, Month, Week, DayOfWeek, Quarter  
2. Fill missing numerics with median; categoricals with mode  
3. Remove outliers on `Units Sold` using 1st–99th percentile  
4. Label-encode: Category, Region, Weather Condition, Seasonality  
5. Create `Discount_frac = Discount / 100`

In [ ]:
clean = preprocess_data(raw)
print('After preprocessing:', clean.shape)
print('New columns:', [c for c in clean.columns if c not in raw.columns])

## 3. Feature Engineering

Steps in `utils/feature_engineering.py`:
- `Effective Price` = Price × (1 − Discount/100)
- `Revenue` = Effective Price × Units Sold
- `Cost Price` = Price × 0.70 (assumed 70 % cost ratio)
- `Profit` = (Effective Price − Cost Price) × Units Sold
- `Profit Margin %`
- `Competitor Diff` = Price − Competitor Pricing
- `Price vs Competitor` — categorical tag (Above/At/Below Market)
- `Demand Category` — Low / Medium / High based on quantiles
- `Stock Turnover` = Units Sold / Inventory Level
- `Discount Effectiveness` = Units Sold / (Discount + 1)

In [ ]:
df = add_business_features(clean)
print('Final shape:', df.shape)
print('Engineered columns:', [c for c in df.columns if c not in clean.columns])

## 4. Exploratory Data Analysis

In [ ]:
# 4a — Price distribution & revenue share by category
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cat, grp in df.groupby('Category'):
    axes[0].hist(grp['Price'], bins=30, alpha=0.6, label=cat)
axes[0].set_title('Price Distribution by Category')
axes[0].set_xlabel('Price (₹)')
axes[0].legend()

cat_rev = df.groupby('Category')['Revenue'].sum()
axes[1].pie(cat_rev, labels=cat_rev.index, autopct='%1.1f%%', startangle=140)
axes[1].set_title('Revenue Share by Category')

plt.tight_layout()
plt.show()

In [ ]:
# 4b — Units Sold by Category × Seasonality
demand_grp = df.groupby(['Category', 'Seasonality'])['Units Sold'].sum().reset_index()
pivot = demand_grp.pivot(index='Category', columns='Seasonality', values='Units Sold').fillna(0)
pivot.plot(kind='bar', figsize=(12, 5), colormap='tab10')
plt.title('Units Sold by Category and Seasonality')
plt.ylabel('Total Units Sold')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# 4c — Promotion impact on average units sold
promo = df.groupby(['Category', 'Holiday/Promotion'])['Units Sold'].mean().reset_index()
promo['Promotion'] = promo['Holiday/Promotion'].map({0: 'No Promo', 1: 'Promotion'})
pivot_p = promo.pivot(index='Category', columns='Promotion', values='Units Sold').fillna(0)
pivot_p.plot(kind='bar', figsize=(12, 5), color=['#7a8499', '#3fb950'])
plt.title('Avg Units Sold: Promotion vs No Promotion')
plt.ylabel('Avg Units Sold')
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

In [ ]:
# 4d — Correlation heatmap (numeric columns used in the app's dashboard)
num_cols = ['Price', 'Discount', 'Competitor Pricing', 'Inventory Level',
            'Units Sold', 'Demand Forecast', 'Revenue', 'Profit', 'Profit Margin %', 'Stock Turnover']
available = [c for c in num_cols if c in df.columns]
corr = df[available].corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, square=True)
plt.title('Correlation Matrix — Retail Variables')
plt.tight_layout()
plt.show()

## 5. Price Elasticity of Demand (PED)

**Formula (log-log regression):**
$$\ln(Q) = \alpha + \beta \cdot \ln(P) \implies \beta = \text{PED}$$

- |PED| > 1 → **Elastic** (demand very sensitive to price)  
- |PED| < 1 → **Inelastic** (demand less sensitive to price)

In [ ]:
# 5a — Category-level elasticity (same function used by elasticity.py view)
cat_elast = compute_price_elasticity(df)
print(cat_elast.to_string(index=False))

In [ ]:
# 5b — Bar chart of elasticity by category
colors = ['#f87171' if s == 'Elastic' else '#4fd1c5' for s in cat_elast['Sensitivity']]
plt.figure(figsize=(10, 5))
bars = plt.bar(cat_elast['Category'], cat_elast['Elasticity'], color=colors)
plt.axhline(y=-1, linestyle='--', color='#f6ad55', label='Elastic boundary (E = -1)')
plt.title('Price Elasticity of Demand by Category')
plt.ylabel('Elasticity coefficient')
plt.legend()
for bar, val in zip(bars, cat_elast['Elasticity']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() - 0.05,
             f'{val:.3f}', ha='center', va='top', fontsize=9, color='white')
plt.tight_layout()
plt.show()

In [ ]:
# 5c — Product-level elasticity (top 20 products by volume)
prod_elast = compute_product_elasticity(df)
print(f'Products analysed: {len(prod_elast)}')
prod_elast.head(10)

## 6. Machine Learning Model Training & Evaluation (CO3)

Three models are trained:

| Model | Scaling |
|-------|--------|
| Linear Regression | Yes (StandardScaler) |
| Decision Tree | No |
| Random Forest | No |

**Target:** `Units Sold`  
**Split:** 80 % train / 20 % test, `random_state=42`

In [ ]:
# Build feature matrix — identical to what the Streamlit app uses
X, y, feat_cols = get_feature_matrix(df)
print('Feature columns:', feat_cols)
print('X shape:', X.shape, '  y shape:', y.shape)

In [ ]:
from sklearn.linear_model    import LinearRegression
from sklearn.tree            import DecisionTreeRegressor
from sklearn.ensemble        import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics         import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing   import StandardScaler

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

models = {
    'Linear Regression': (LinearRegression(), True),
    'Decision Tree':     (DecisionTreeRegressor(max_depth=10, random_state=42), False),
    'Random Forest':     (RandomForestRegressor(n_estimators=60, max_depth=12, random_state=42, n_jobs=-1), False),
}

rows = []
trained = {}   # store fitted models for later use
for name, (model, use_sc) in models.items():
    Xtr = X_train_sc if use_sc else X_train
    Xte = X_test_sc  if use_sc else X_test
    model.fit(Xtr, y_train)
    yp  = model.predict(Xte)
    r2  = round(r2_score(y_test, yp), 4)
    mae = round(mean_absolute_error(y_test, yp), 2)
    rmse = round(mean_squared_error(y_test, yp)**0.5, 2)
    rows.append({'Model': name, 'R²': r2, 'MAE': mae, 'RMSE': rmse})
    trained[name] = {'model': model, 'y_pred': yp, 'scaled': use_sc}
    print(f'{name:<22} R²={r2}  MAE={mae}  RMSE={rmse}')

results_df = pd.DataFrame(rows)
results_df

In [ ]:
# Model comparison bar charts
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = ['#6e76e5', '#4fd1c5', '#b794f4']

for ax, metric in zip(axes, ['R²', 'MAE', 'RMSE']):
    bars = ax.bar(results_df['Model'], results_df[metric],
                  color=palette[:len(results_df)])
    ax.set_title(f'{metric} Comparison')
    ax.set_xticklabels(results_df['Model'], rotation=20, ha='right', fontsize=9)
    for bar, val in zip(bars, results_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(results_df[metric]) * 0.01,
                f'{val:.4f}', ha='center', fontsize=8)

plt.suptitle('Model Performance Comparison — DemandLens', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Actual vs Predicted scatter — best model by R²
best_name = results_df.loc[results_df['R²'].idxmax(), 'Model']
y_pred_best = trained[best_name]['y_pred']

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_best, alpha=0.4, color='#6e76e5', s=15)
lims = [min(y_test.min(), y_pred_best.min()),
        max(y_test.max(), y_pred_best.max())]
plt.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect fit')
plt.title(f'Actual vs Predicted — {best_name}')
plt.xlabel('Actual Units Sold')
plt.ylabel('Predicted Units Sold')
plt.legend()
plt.tight_layout()
plt.show()

## 7. Price Optimisation Demo (CO2)

Uses the best-performing model (Random Forest) to find the revenue-maximising price point for a sample product configuration — identical logic to `views/optimizer.py`.

In [ ]:
best_model  = trained[best_name]['model']
scaled_flag = trained[best_name]['scaled']

# Base config: first row from test set
base = X_test.iloc[0].to_dict()
current_price = base['Price']

# Grid search over ±50% of current price
prices   = np.linspace(current_price * 0.5, current_price * 1.5, 100)
revenues = []

for p in prices:
    row = base.copy()
    row['Price'] = p
    inp = pd.DataFrame([row])[feat_cols]
    if scaled_flag:
        demand = max(0, float(best_model.predict(scaler.transform(inp))[0]))
    else:
        demand = max(0, float(best_model.predict(inp)[0]))
    # effective price after existing discount fraction
    eff_price = p * (1 - base.get('Discount_frac', 0))
    revenues.append(eff_price * demand)

opt_idx   = int(np.argmax(revenues))
opt_price = prices[opt_idx]
print(f'Current price : ₹{current_price:.2f}')
print(f'Optimal price : ₹{opt_price:.2f}')
print(f'Max revenue   : ₹{revenues[opt_idx]:,.0f}')

In [ ]:
# Revenue optimisation curve
plt.figure(figsize=(11, 5))
plt.plot(prices, revenues, color='#6e76e5', linewidth=2)
plt.fill_between(prices, revenues, alpha=0.10, color='#6e76e5')
plt.axvline(x=current_price, color='#f687b3', linestyle=':', linewidth=1.8,
            label=f'Current ₹{current_price:.0f}')
plt.axvline(x=opt_price, color='#3fb950', linestyle='--', linewidth=1.8,
            label=f'Optimal ₹{opt_price:.0f}')
plt.title(f'Revenue Optimisation Curve — {best_name}', fontsize=13)
plt.xlabel('Price (₹)')
plt.ylabel('Predicted Revenue (₹)')
plt.legend()
plt.grid(alpha=0.2)
plt.tight_layout()
plt.show()